# Qualifire — Dashboards via matplotlib + plotly

Reads the **system table** and produces visualizations at two
granularities:

- **Executive view** (sections 3–4) — overall pass rate, daily trend,
  worst offenders. One scroll, one printable page; great for an
  exec summary or a product pitch slide.
- **Data-steward view** (sections 5–7) — per-dataset breakdown,
  per-validation drill-down with metric history and threshold lines,
  severity-coloured time series, distribution shape.

Each section pairs a **matplotlib** rendering (static, print-friendly,
PDF-ready) with a **plotly** rendering (interactive, hover tooltips,
zoomable). Pick whichever fits the audience.

**Every cell below calls public APIs from `qualifire.reporting`** —
the notebook itself doesn't invent any rendering logic. That makes it
equally useful as a worked example (end users can copy-paste the same
calls into their own projects) and a smoke test of the API surface.

Backend selection is configurable: `'sqlite'` for local dev,
`'external_catalog'` / `'delta'` / `'jdbc'` for production. Same
cells, different backends — see section 1.

## 1. Configure backend + output directory

**Edit the next cell directly.** AIDP Workbench has no terminal, so
the constants in the cell are the primary configuration path. The
env-var fallbacks (`QF_BACKEND`, `QF_SYSTEM_TABLE`, …) are a secondary
path for local shells that have one.

The matplotlib charts are saved as PNG files under `OUTPUT_DIR`;
plotly charts render inline.

In [ ]:
import os, tempfile
from pathlib import Path

# ---------------------------------------------------------------------------
# Edit the constants in THIS CELL to match your environment.
# (Env vars work too, but they're a fallback — AIDP Workbench has no
# terminal, so editing the cell is the primary path.)
# ---------------------------------------------------------------------------

# Pick one of: 'sqlite' | 'external_catalog' | 'delta' | 'jdbc'
#   sqlite           → local dev, file-based
#   external_catalog → AIDP Workbench (Oracle ADW or any catalog-resident
#                      table — Hive / Unity Catalog)
#   delta            → Delta Lake table
#   jdbc             → generic JDBC datasource (caller supplies driver)
BACKEND = os.environ.get('QF_BACKEND', 'sqlite')

# Where the system table lives — see dashboard_html.ipynb section 1
# for the full per-backend table.
SYSTEM_TABLE = os.environ.get(
    'QF_SYSTEM_TABLE',
    str(Path(tempfile.gettempdir()) / 'qualifire_history.sqlite'),
)

# Where to drop the generated PNG / HTML chart exports.
OUTPUT_DIR = Path(os.environ.get('QF_OUTPUT_DIR', tempfile.mkdtemp(prefix='qf_charts_')))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Default lookback (days).
DAYS = int(os.environ.get('QF_DAYS', '90'))

# Required only when BACKEND == 'jdbc'. Generic JDBC example:
#   url:      'jdbc:oracle:thin:@//<host>:<port>/<service_name>'
#   user:     DB user
#   password: DB password
#   driver:   'oracle.jdbc.OracleDriver'
JDBC_CONFIG = {
    'url':      os.environ.get('QF_JDBC_URL', ''),
    'user':     os.environ.get('QF_JDBC_USER', ''),
    'password': os.environ.get('QF_JDBC_PASSWORD', ''),
    'driver':   os.environ.get('QF_JDBC_DRIVER', 'oracle.jdbc.OracleDriver'),
}

print(f'BACKEND      : {BACKEND}')
print(f'SYSTEM_TABLE : {SYSTEM_TABLE}')
print(f'OUTPUT_DIR   : {OUTPUT_DIR}')
print(f'DAYS         : {DAYS}')
if BACKEND == 'jdbc':
    print(f"JDBC URL     : {JDBC_CONFIG['url'] or '(not set — fill in JDBC_CONFIG above)'}")

## 2. Load — system table → pandas DataFrame

Two API calls do all the work:

* `make_storage(...)` opens a backend-agnostic
  [`SystemTableStorage`](../../qualifire/storage/base.py) handle.
* `load_health_dataframe(storage, days=N)` returns a coerced pandas
  DataFrame — typed `run_timestamp` and `partition_ts`, numeric
  `metric_value`, derived `day` and uppercase `severity`. All charts
  below consume `df`.

In [ ]:
from qualifire.reporting import make_storage, load_health_dataframe

storage = make_storage(
    BACKEND, SYSTEM_TABLE,
    JDBC_CONFIG if BACKEND == 'jdbc' else None,
)
df = load_health_dataframe(storage, days=DAYS)
print(f'Loaded {len(df)} rows for the last {DAYS} days.')
df.head()

## 3. Executive summary — matplotlib (print-friendly)

Three things any exec wants to see in 30 seconds: **how healthy are
we?** (overall pass rate), **is it trending right?** (daily pass rate
over time), **what's broken right now?** (worst offenders by error
count). The function returns a matplotlib `Figure` and, when
`output_dir=` is given, also writes `executive_summary.png`.

In [ ]:
from qualifire.reporting import plot_executive_summary

plot_executive_summary(df, output_dir=OUTPUT_DIR)

## 4. Executive summary — plotly (interactive)

Same data, hover-able. Donut shows the severity mix; the line shows
the trend. Lighter weight than the full interactive dashboard
(`generate_interactive_html`) and easy to embed in a Confluence/wiki
page via `fig.to_html(...)`.

In [ ]:
from qualifire.reporting import plot_executive_summary_interactive

fig = plot_executive_summary_interactive(df, days=DAYS)
if fig is not None:
    fig.show()
else:
    print('No data — skip.')

## 5. Data-steward view — heatmap (matplotlib)

Cells of `dataset × day` coloured by worst severity per cell. One
look tells you *which* dataset is failing *when*. Density-respecting
layout — fine for many datasets and 30+ days.

In [ ]:
from qualifire.reporting import plot_dataset_day_heatmap

plot_dataset_day_heatmap(df, output_dir=OUTPUT_DIR)

## 6. Data-steward view — per-validation history (plotly)

Pick one `(dataset, validation)` pair and plot the metric value over
time, coloured by severity, separated per dimension when applicable.
This is what you reach for when investigating a specific failure.

Without args the helper picks the validation with the most ERROR
rows so the demo lands somewhere interesting; pass
`dataset_name=...` and `validation_name=...` to focus elsewhere.

In [ ]:
from qualifire.reporting import plot_validation_history_interactive

fig = plot_validation_history_interactive(df)
if fig is not None:
    fig.show()
else:
    print('No numeric history to plot — skip.')

## 7. Hierarchical breakdown (plotly sunburst + treemap)

`dataset → validation_type → severity` as a sunburst. Click a wedge
to drill in. Useful for mid-level managers who want to see where
errors concentrate without going into single-validation history.
Treemap is rendered alongside for the same data — both are valid;
audiences differ.

In [ ]:
from qualifire.reporting import plot_severity_hierarchy

sun, tree = plot_severity_hierarchy(df)
if sun is not None:
    sun.show()
    tree.show()
else:
    print('No data — skip.')

## 8. Severity distribution — matplotlib + plotly side-by-side

Two views of the same severity counts: a stacked bar per validation
type (matplotlib, exportable) and an interactive violin plot of
`metric_value` per severity (plotly). Together they show *what kinds
of checks fire* and *what magnitudes* the failing ones report.

In [ ]:
from qualifire.reporting import (
    plot_severity_by_type,
    plot_metric_distribution_by_severity,
)

plot_severity_by_type(df, output_dir=OUTPUT_DIR)

In [ ]:
fig = plot_metric_distribution_by_severity(df)
if fig is not None:
    fig.show()
else:
    print('No numeric metric_value rows for the violin plot.')

## 9. Files written

PNG snapshots of the matplotlib charts live in `OUTPUT_DIR`. Plotly
charts are interactive in the notebook — call `fig.write_html(path)`
or `fig.write_image(path)` (needs `kaleido`) on any of them above to
save them out.

In [ ]:
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p}  ({p.stat().st_size:,} bytes)')